# Dataset & DataLoader

Wrap synthetic data in a **`Dataset`** and batch it with **`DataLoader`** for efficient training loops.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


## 1. Generate synthetic data

In [ ]:
gen = DummyDataGenerator(n_samples=128, n_features=8, n_classes=3)
X, y = gen.tensors()
ds = TabularDataset(X, y)
print(f"Dataset length: {len(ds)}, sample X shape: {ds[0][0].shape}")

## 2. DataLoader — batching & shuffling

In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=True)
batch_x, batch_y = next(iter(loader))
print(f"Batch X: {batch_x.shape}, Batch y: {batch_y.shape}")

## 3. Visualize one batch — feature heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(batch_x.numpy(), aspect='auto', cmap='viridis')
ax.set_title('One batch of features (16 × 8)'); ax.set_xlabel('feature')
plt.tight_layout(); plt.show()

## 4. Class distribution in batch vs full dataset

In [ ]:
full_counts = torch.bincount(y, minlength=3).numpy()
batch_counts = torch.bincount(batch_y, minlength=3).numpy()
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3); w = 0.35
ax.bar(x - w/2, full_counts, w, label='full dataset', color='steelblue')
ax.bar(x + w/2, batch_counts, w, label='one batch', color='coral')
ax.set_xticks(x); ax.set_xlabel('class'); ax.legend()
ax.set_title('Label distribution')
plt.tight_layout(); plt.show()

## 5. Iterate multiple batches

In [ ]:
batch_sizes = [len(by) for bx, by in loader]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(batch_sizes)), batch_sizes, color='seagreen', edgecolor='black')
ax.set_xlabel('batch index'); ax.set_ylabel('batch size')
ax.set_title(f'DataLoader yields {len(batch_sizes)} batches (batch_size=16)')
plt.tight_layout(); plt.show()